[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RDC28/vl-gguf-lab/blob/fm/vlgguf-colab/notebooks/05_remove_layers.ipynb)

# Remove GGUF layers

Remove each predeclared block in the F16 control and renumber the remainder. Score every variant against the unedited control on the same held-out scenes.

Choose **Runtime > Change runtime type > T4 GPU** (or L4/A100), then run the cells in order. Setup reads a GitHub token from Colab Secrets for this private repository. Drive access is requested by the run cell. Set `VL_LAB_SMOKE=1` before setup for a bounded CPU check.

In [ ]:
import base64
import getpass
import os
from pathlib import Path
import subprocess
import sys

SMOKE = os.environ.get("VL_LAB_SMOKE", "0") == "1"
REPO_REF = "fm/vlgguf-colab"  # Use "main" after this branch has merged.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content/vl-gguf-lab")
    if not checkout.exists():
        # Private repository: add GH_TOKEN to Colab Secrets with repository read access.
        from google.colab import userdata
        try:
            token = userdata.get("GH_TOKEN")
        except Exception:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        env = os.environ.copy()
        auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0="Authorization: Basic " + auth)
        subprocess.run(["git", "clone", "--branch", REPO_REF, "--depth", "1",
                        "https://github.com/RDC28/vl-gguf-lab.git", str(checkout)], env=env, check=True)
        del token, auth, env
    os.chdir(checkout)
else:
    checkout = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(checkout))
from src.lab_setup import setup
setup(smoke=SMOKE)


## Configuration

Use the same settings to resume. Saved run folders are printed before inference starts. Set `RESUME` to that folder name after reconnecting; outputs and optimizer checkpoints are recovered from Drive.

In [ ]:
from src.lab_experiments import configuration, run_case, show_summary

CONFIG = configuration(5, smoke=SMOKE)
# Edit model_tier: "t4", "l4", "a100" (case 1 only).
# Cases 3-6 deliberately use a small F16 control, not the VL weights.
# CONFIG["drive_root"] = "MyDrive/vl-gguf-lab/results"
# Full runs: leave limit=None; max_tokens=4096; epochs=3.
RESUME = None  # Paste a saved run folder name, keeping the same CONFIG.
LOCAL_ROOT = os.environ.get("VL_LAB_RESULTS")  # Defaults to results/ outside Colab.
print(CONFIG)


In [ ]:
result = run_case(5, CONFIG, resume=RESUME, root=LOCAL_ROOT)


## Measured summary

The charts below are computed from saved outputs. A capped answer counts as truncated even if it has useful text. Word overlap and parsed structure are proxies, so inspect the raw answers before making a quality claim. Smoke outputs are setup checks only. All Colab result numbers in the paper remain pending until a full run is completed.

In [ ]:
show_summary(result)
